# 02 — Análisis de Características

**Objetivos:**
- Matriz de correlación entre features de regresión y clasificación
- Perfiles Likert para grupos de variables
- Análisis de desbalance de clases
- Revisión de variables con alta correlación con los targets

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load import load_raw
from src.data.clean import encode_binary_targets
from src.config import (
    REGRESSION_FEATURE_COLS, CLASSIFICATION_FEATURE_COLS,
    CLASSIFICATION_ALL_TARGETS, CLASSIFICATION_TARGET_DEFAULT
)
from src.visualization.plots import plot_correlation_matrix, plot_likert_profiles

sns.set_theme(style='whitegrid')
%matplotlib inline

df = load_raw('../data/SLV2013_Public_Use.csv')
df = encode_binary_targets(df, CLASSIFICATION_ALL_TARGETS)

## 1. Correlación — Features de regresión

In [ ]:
fig = plot_correlation_matrix(df, REGRESSION_FEATURE_COLS)
plt.show()

## 2. Correlación con target de clasificación

In [ ]:
available = [c for c in CLASSIFICATION_FEATURE_COLS if c in df.columns]
target = CLASSIFICATION_TARGET_DEFAULT

corr_with_target = df[available + [target]].corr(method='spearman')[target].drop(target).sort_values()

fig, ax = plt.subplots(figsize=(8, 10))
corr_with_target.plot.barh(ax=ax, color=corr_with_target.map(lambda x: 'steelblue' if x > 0 else 'salmon'))
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Correlación Spearman con {target}')
ax.set_xlabel('Correlación')
plt.tight_layout()
plt.show()

## 3. Perfiles Likert — Dieta

In [ ]:
diet_cols = ['Q10', 'Q11', 'Q12', 'Q13', 'Q14']
fig = plot_likert_profiles(df, diet_cols, title='Distribución de respuestas — Dieta')
plt.show()

## 4. Balance de clases

In [ ]:
imbalance = []
for t in CLASSIFICATION_ALL_TARGETS:
    if t in df.columns:
        vc = df[t].value_counts(normalize=True)
        majority = vc.max()
        imbalance.append({'target': t, 'majority_class_pct': round(majority * 100, 1)})

pd.DataFrame(imbalance).sort_values('majority_class_pct', ascending=False)

## 5. Features de ingeniería

In [ ]:
from src.features.engineer import add_all_engineered_features, get_engineered_feature_names
df_eng = add_all_engineered_features(df)
eng_cols = get_engineered_feature_names()
df_eng[eng_cols].describe()